In [46]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [47]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()
    print(len(text))

1115394


In [48]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [171]:
learning_rate = 1e-3

In [49]:
# Metinde yer alan tüm karakterlerin listesi ve uzunluğu

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [50]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello, how are you?"))
print(decode(encode("hello, how are you?")))

[46, 43, 50, 50, 53, 6, 1, 46, 53, 61, 1, 39, 56, 43, 1, 63, 53, 59, 12]
hello, how are you?


In [51]:
# embedding dizisini vektöre dönüştürme
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [52]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [61]:
block_size = 8
print(train_data[:block_size])
print(train_data[1:block_size + 1])

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])


In [56]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input -> {context} target -> {target}")
    print(f"input -> {''.join(itos[c.item()] for c in context)} target -> {itos[target.item()]}")

input -> tensor([18]) target -> 47
input -> F target -> i
input -> tensor([18, 47]) target -> 56
input -> Fi target -> r
input -> tensor([18, 47, 56]) target -> 57
input -> Fir target -> s
input -> tensor([18, 47, 56, 57]) target -> 58
input -> Firs target -> t
input -> tensor([18, 47, 56, 57, 58]) target -> 1
input -> First target ->  
input -> tensor([18, 47, 56, 57, 58,  1]) target -> 15
input -> First  target -> C
input -> tensor([18, 47, 56, 57, 58,  1, 15]) target -> 47
input -> First C target -> i
input -> tensor([18, 47, 56, 57, 58,  1, 15, 47]) target -> 58
input -> First Ci target -> t


In [63]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return (x, y)

xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)
print(xb[1])
print(yb[1])

print('-----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"input: {context.tolist()} target: {target}")
    print('----')

torch.Size([4, 8])
torch.Size([4, 8])
tensor([44, 53, 56,  1, 58, 46, 39, 58])
tensor([53, 56,  1, 58, 46, 39, 58,  1])
-----
input: [24] target: 43
input: [24, 43] target: 58
input: [24, 43, 58] target: 5
input: [24, 43, 58, 5] target: 57
input: [24, 43, 58, 5, 57] target: 1
input: [24, 43, 58, 5, 57, 1] target: 46
input: [24, 43, 58, 5, 57, 1, 46] target: 43
input: [24, 43, 58, 5, 57, 1, 46, 43] target: 39
----
input: [44] target: 53
input: [44, 53] target: 56
input: [44, 53, 56] target: 1
input: [44, 53, 56, 1] target: 58
input: [44, 53, 56, 1, 58] target: 46
input: [44, 53, 56, 1, 58, 46] target: 39
input: [44, 53, 56, 1, 58, 46, 39] target: 58
input: [44, 53, 56, 1, 58, 46, 39, 58] target: 1
----
input: [52] target: 58
input: [52, 58] target: 1
input: [52, 58, 1] target: 58
input: [52, 58, 1, 58] target: 46
input: [52, 58, 1, 58, 46] target: 39
input: [52, 58, 1, 58, 46, 39] target: 58
input: [52, 58, 1, 58, 46, 39, 58] target: 1
input: [52, 58, 1, 58, 46, 39, 58, 1] target: 46
--

In [167]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return (logits, loss)

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
        
        
model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

torch.Size([256, 65])
tensor(4.7821, grad_fn=<NllLossBackward0>)


In [172]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [169]:
batch_size = 32
max_iters = 5000
for iter in range(max_iters):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 200 == 0:
        print(loss.item())

4.648484230041504
4.473455429077148
4.2933831214904785
4.020341873168945
3.8753597736358643
3.7026751041412354
3.6299033164978027
3.3767645359039307
3.435028553009033
3.3145272731781006
3.1231141090393066
3.0234131813049316
2.956467866897583
2.8492226600646973
2.794816493988037
2.7292535305023193
2.682288408279419
2.7295515537261963
2.8177685737609863
2.7024922370910645
2.5700602531433105
2.6373143196105957
2.6741251945495605
2.7019453048706055
2.4774715900421143


In [176]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=600)[0].tolist()))


JUENooreswh w mear al oyouthouguievouse neaulyooresesthowige lathameluriomad.
Hor ss heathy hithak samie; gol tt theil gnove Youifigulis; CEnd, tait.
D LINTh ithithyoshit ceatheten n wen y'Fowa men orseedize thaclily, d tounathe andwhoug, ce blave ILORO: so torvertoneind ckeed,
DWirnd
'st, ourje t late than sorowissokefowe es wrswe be, ay,
r fr thism tseamouslelle

ADYO: thaity me,

I tous, thourtheshel hirt stof ispanethit a wind!
Kirs anthmulelanithat rse y.
ILAUENam: o nisting le litcho O:

Whethistoss y o hounourct wek horsof len wil; ad Gurs I nt y pat trener:
UThof selespss s ty rncigomo


In [97]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [114]:
# 1. ağırlıklı temsil oluşturma

xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t, C)
        xbow[b,t] = torch.mean(xprev, 0)

x[0, :2], xbow[0]

(tensor([[ 0.1808, -0.0700],
         [-0.3596, -0.9152]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [122]:
# 2. ağırlıklı temsil oluşturma

# Her token için kendisinden önce gelen tokenlerin bilgilerini kullanarak
# ağırlıklı bir temsil oluşturmak istiyoruz.
# Gelecekteki tokenlerin görülmesini istemiyoruz (causal attention).
# Bu nedenle torch.tril kullanarak gelecekteki tokenlerin ağırlıklarını 0 yapıyoruz.

wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-7)

True

In [118]:
import math

math.exp(0), math.exp(float('-inf'))

(1.0, 0.0)

In [123]:
# 3. ağırlıklı temsil oluşturma

# Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
# e^-inf = 0
# e^0 = 1
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow2, xbow3, atol=1e-7)

True

In [154]:
# 4. ağırlıklı temsil oluşturma: self-attention
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ----> (B, T, T)

tril = torch.tril(torch.ones((T,T)))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei_n = F.softmax(wei, dim=-1)
wei_d = F.softmax(wei * (head_size**-0.5), dim=-1)

v = value(x)
out = wei_n @ v
# out = wei @ x

print(wei_n[0])
print(wei_d[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
   

In [146]:
xt = torch.tensor([0.1, 0.1, 0.4]).float()
prob = F.softmax(xt, dim=0)
prob

tensor([0.2985, 0.2985, 0.4030])

In [170]:
eval_iters = 5000

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print(estimate_loss())

{'train': tensor(2.5602), 'val': tensor(2.5673)}
